In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))


In [2]:
import gradio as gr
from Day4LLMCalling import Llms, tools
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')


# openai = OpenAI(base_url="https://openrouter.ai/api/v1",api_key=os.getenv('OPENROUTER_API_KEY'))

In [3]:
import requests
from PIL import Image
from io import BytesIO
import base64
# Assuming 'client' is your OpenAI client initialized with OpenRouter's base_url

def artist(city):
    if not city:
        return None
    
    print(f"Generating image for: {city}")

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}",
            "Content-Type": "application/json",
        },
        json={
            "model": "openai/gpt-5-image-mini",
            "messages": [{"role": "user", "content": f"generate an artistic image for {city}"}],
            "modalities": ["image", "text"],
        },
        timeout=120,
    )

    result = response.json()
    message = result['choices'][0]["message"]
    images = message.get("images", [])
    if images:
        img_url = images[0]["image_url"]["url"]       # data:image/png;base64,...
        image_base64 = img_url.split(",")[1]
        image_data = base64.b64decode(image_base64)
        print("Image generated!")
        return Image.open(BytesIO(image_data))
    else:
        print("No image returned")
        return None


In [8]:
from pydub import AudioSegment


def talker(text):
    if not text:
        return None

    print(f"Generating audio for: {text}")

    response = requests.post(
        "https://openrouter.ai/api/v1/audio/speech",
        headers={
            "Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}",
            "Content-Type": "application/json",
        },
        json={
            "model": "openai/gpt-4o-mini-tts-2025-12-15",
            "voice": "onyx",
            "input": text,
            "response_format": "mp3",
        },
        timeout=60,
    )

    if response.status_code != 200:
        print(f"Error: {response.json()}")
        return None

    # Save raw bytes directly — no ffmpeg needed
    output_path = "output.mp3"
    with open(output_path, "wb") as f:
        f.write(response.content)

    print("Audio generated!")
    return output_path  # Gradio's Audio component accepts a file path

Input tokens: 339
Output tokens: 218
Total tokens: 557
Total calculated cost: 0.0340 cents
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'openai.types.chat.chat_completion_message.ChatCompletionMessage'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
[{'role': 'system', 'content': ''}, {'role': 'user', 'content': 'hi there'}, {'role': 'assistant', 'content': 'Hi! 👋 How can I help you today?'}, {'role': 'user', 'content': 'what is the price for berlin?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_u2UDYuvXrtadM4nW9daaYI05', function=Function(arguments='{"destination_city":"Berlin","price":0,"action":"get_by_name"}', name='set_ticket_price'), type='function', index=0)], reasoning=None), {'role': 'tool', 'content': "The result of your query is {'city': 'berlin', 'price': 499}", 'tool_call_id': 'call_u2UDYuvXrtadM4nW9daaYI05

Traceback (most recent call last):
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\blocks.py", line 2192, in process_api
    data = await self.postprocess_data(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\blocks.py", line 1955, in postprocess_data
    prediction_value = await anyio.to_thread.run_sync(
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\anyio\to_thread.p

In [ ]:
from Day4LLMCalling.messageSeries import mSeries
import openai.types.chat.chat_completion_message as msg

def wrapLlm(message):
    response,tool_arguments = Llms.callModel(message, source='openai',tools=tools,return_tool_arguments=True)
    history = mSeries.promptList.get(0,{}).get('gpt-5.4-nano',[])
    gradio_history = []
    for item in history:
        # print(type(item))
        if not (isinstance(item, msg.ChatCompletionMessage) or item.get('role') in ['tool','system'] ):
            gradio_history.append(item)
    # print(history)
    # print(gradio_history)
    if len(tool_arguments)==0:
        return response, gradio_history, None
    return response, gradio_history, tool_arguments[0]['destination_city']

In [14]:
with gr.Blocks() as ui:
    city_state = gr.State()
    audio_state = gr.State()
    with gr.Row():
        chat_history = gr.Chatbot(height=500, label='Chat History')
        image_box = gr.Image(height=500, interactive=False, show_label=False)
    with gr.Row():
        audio_box = gr.Audio(autoplay=True, type='filepath')
    with gr.Row():
        message_box = gr.Textbox(label='Chat with AI')

        message_box.submit(
            fn = wrapLlm,
            inputs=[message_box],
            outputs=[audio_state,chat_history,city_state]
            ).then(
            fn=artist,
            inputs=[city_state], 
            outputs=[image_box]
            ).then(
            fn=talker,
            inputs=[audio_state],
            outputs=[audio_box]
            )

ui.launch()
        

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Input tokens: 699
Output tokens: 15
Total tokens: 714
Total calculated cost: 0.0159 cents
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'openai.types.chat.chat_completion_message.ChatCompletionMessage'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
[{'role': 'system', 'content': ''}, {'role': 'user', 'content': 'hi there'}, {'role': 'assistant', 'content': 'Hi! 👋 How can I help you today?'}, {'role': 'user', 'content': 'what is the price for berlin?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_u2UDYuvXrtadM4nW9daaYI05', function=Function(arguments='{"destination_city":"Berlin","price":0,"action":"get_by_name"}', name='set_ticket_price'), type='function', index=0)], reason

Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Users\PGCP-AI\AppData\Local\Programs\Python\Python312\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\PGCP-AI\AppData\Local\Programs\Python\Python312\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host
Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Users\PGCP-AI\AppData\Local\Programs\Python\Python312\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\PGCP-AI\AppData\Local\Pro

Audio generated!


In [7]:
mSeries.promptList

{}

Input tokens: 220
Output tokens: 15
Total tokens: 235
Total calculated cost: 0.0063 cents
<class 'dict'>
<class 'dict'>
<class 'dict'>
[{'role': 'system', 'content': ''}, {'role': 'user', 'content': 'hi there'}, {'role': 'assistant', 'content': 'Hi! 👋 How can I help you today?'}]
[{'role': 'system', 'content': ''}, {'role': 'user', 'content': 'hi there'}, {'role': 'assistant', 'content': 'Hi! 👋 How can I help you today?'}]
Generating audio for: Hi! 👋 How can I help you today?
Audio generated!


Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Users\PGCP-AI\AppData\Local\Programs\Python\Python312\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\PGCP-AI\AppData\Local\Programs\Python\Python312\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host


called get_by_name on berlin
Input tokens: 309
Output tokens: 16
Total tokens: 325
Total calculated cost: 0.0082 cents
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'openai.types.chat.chat_completion_message.ChatCompletionMessage'>
<class 'dict'>
<class 'dict'>
[{'role': 'system', 'content': ''}, {'role': 'user', 'content': 'hi there'}, {'role': 'assistant', 'content': 'Hi! 👋 How can I help you today?'}, {'role': 'user', 'content': 'what is the price for berlin?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_u2UDYuvXrtadM4nW9daaYI05', function=Function(arguments='{"destination_city":"Berlin","price":0,"action":"get_by_name"}', name='set_ticket_price'), type='function', index=0)], reasoning=None), {'role': 'tool', 'content': "The result of your query is {'city': 'berlin', 'price': 499}", 'tool_call_id': 'call_u2UDYuvXrtadM4nW9daaYI05'}

In [9]:
mSeries.promptList

{0: {'gpt-5.4-nano': [{'role': 'system', 'content': ''},
   {'role': 'user', 'content': 'hi there'},
   {'role': 'assistant', 'content': 'Hi! 👋 How can I help you today?'},
   {'role': 'user', 'content': 'what is the price for berlin?'},
   ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_u2UDYuvXrtadM4nW9daaYI05', function=Function(arguments='{"destination_city":"Berlin","price":0,"action":"get_by_name"}', name='set_ticket_price'), type='function', index=0)], reasoning=None),
   {'role': 'tool',
    'content': "The result of your query is {'city': 'berlin', 'price': 499}",
    'tool_call_id': 'call_u2UDYuvXrtadM4nW9daaYI05'},
   {'role': 'assistant', 'content': 'The price for **Berlin** is **$499**.'},
   {'role': 'user', 'content': 'what else do you know about berlin?'},
   {'role': 'assistant',
    'content': 'Here are a few quick highlights about **Berlin*

Input tokens: 565
Output tokens: 25
Total tokens: 590
Total calculated cost: 0.0144 cents
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'openai.types.chat.chat_completion_message.ChatCompletionMessage'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
[{'role': 'system', 'content': ''}, {'role': 'user', 'content': 'hi there'}, {'role': 'assistant', 'content': 'Hi! 👋 How can I help you today?'}, {'role': 'user', 'content': 'what is the price for berlin?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_u2UDYuvXrtadM4nW9daaYI05', function=Function(arguments='{"destination_city":"Berlin","price":0,"action":"get_by_name"}', name='set_ticket_price'), type='function', index=0)], reasoning=None), {'role': 'tool', 'content': "The result of your query is {'city': 'berlin', 'price': 499}", 'tool_call_id': '

Traceback (most recent call last):
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\blocks.py", line 2192, in process_api
    data = await self.postprocess_data(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\blocks.py", line 1955, in postprocess_data
    prediction_value = await anyio.to_thread.run_sync(
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\anyio\to_thread.p